# AfroLuo — Entraînement du modèle de traduction français → ewondo
Notebook de documentation et de reproduction. **Auteure : Noumedem Nerge (backend & IA, équipe AfroLuo).**
Projet : SAFINUM Innovate Challenge — extension Chrome qui affiche l'interface *Safinum Pay* en ewondo.

## 1. Ce qui a été fait (vue d'ensemble)
| Étape | Contenu | Résultat |
|---|---|---|
| 1 | Données : phrases fr↔ewondo, séparées en train / val / test **sans fuite** (`data_v4/`) + glossaire validé (`glossaire_utilise_1.csv`) | jeux `fr_ew_train/val/test/test_lexique` |
| 2 | Modèle 1 : **MarianMT** (Helsinki-NLP/opus-mt, 75 M paramètres), 4 directions | v2 (test de 664 phrases) : fr→ew BLEU **27,4**, ew→fr **25,9**, en→ew **15,3**, ew→en **14,2** ; v3 fr→ew sur 715 exemples jamais vus : BLEU **19,7**, chrF++ **42,1** |
| 3 | Modèle 2 : **NLLB-200-distilled-600M** affiné (ewondo mappé sur le code `lin_Latn`) | chrF++ **45,1** (meilleur que Marian) |
| 4 | Comparaison des deux modèles sur le même test + taux de termes du glossaire retrouvés | NLLB retenu comme meilleur |
| 5 | Compression CTranslate2 int8 pour un usage local | à exécuter (section 7) |
| 6 | Utilisation : le modèle **propose** des traductions, la locutrice native (Philomène) les valide, puis elles entrent dans le dictionnaire de l'extension | l'extension fonctionne hors ligne |

**Important (honnêteté scientifique).** Le modèle ne tourne *pas* dans l'extension. Les scores (chrF++ ≈ 45) montrent que la traduction automatique de l'ewondo reste imparfaite : on s'en sert comme *assistant de traduction*, jamais pour les montants, frais ou bénéficiaires, et toute phrase critique est validée par une locutrice native.

Environnement : Google Colab, GPU T4, transformers + datasets + sacrebleu.

## 2. Installation et accès à Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # si besoin : drive.mount('/content/drive', force_remount=True)
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

## 3. Données
Dossier `afroluo_v3/` sur Drive :
- `data_v4/fr_ew_train.jsonl`, `fr_ew_val.jsonl`, `fr_ew_test.jsonl`, `fr_ew_test_lexique.jsonl` — une ligne = `{"src": "...", "tgt": "..."}`.
- `glossaire_utilise_1.csv` — termes validés (colonnes « Entrée du lexique », « Éwondo », « Page »). Ajouté **8 fois** au train pour qu'il pèse sans dominer.
- `traduction_francais_ewondo.csv` — textes de l'interface, pour mesurer le vocabulaire.

Règles anti-fuite : aucune phrase du test/validation n'est dans le train ; le glossaire n'est ajouté qu'au train.

In [ ]:
import json, os
B = "/content/drive/MyDrive/afroluo_v3"; DATA = B + "/data_v4"
for part in ("train","val","test","test_lexique"):
    p = f"{DATA}/fr_ew_{part}.jsonl"
    if os.path.exists(p):
        n = sum(1 for _ in open(p, encoding="utf-8")); print(part, n, "exemples")
print(open(f"{DATA}/fr_ew_train.jsonl", encoding="utf-8").readline())

## 4. Modèle 1 — MarianMT (Helsinki-NLP/opus-mt)
**Principe** : on part d'un modèle de traduction déjà entraîné sur d'autres langues (*transfer learning*) et on l'affine sur nos paires avec l'ewondo.

| Direction | Modèle de départ | Données (train / val / test) | BLEU test |
|---|---|---|---|
| fr → ew | `Helsinki-NLP/opus-mt-fr-en` | 11 273 / 1 326 / 664 | **27,4** |
| ew → fr | `Helsinki-NLP/opus-mt-en-fr` | 11 273 / 1 326 / 664 | **25,9** (24,97 sur une autre exécution) |
| en → ew | `Helsinki-NLP/opus-mt-en-fr` | 1 843 / 217 / 109 | **15,3** |
| ew → en | `Helsinki-NLP/opus-mt-fr-en` | 1 843 / 217 / 109 | **14,2** |

**Réglages (identiques pour les 4 directions)** : 15 époques maximum, batch 16, lr 5e-5, warmup 200 pas, weight decay 0,01, fp16, `max_length` 128, évaluation et sauvegarde à chaque époque, meilleur modèle selon le BLEU, arrêt anticipé (patience 3), sauvegarde complète du modèle sur Drive à chaque époque (utile quand Colab coupe).

**Historique**
1. Première version (`afroluo_data_ready.zip` : 4 119 / 485 / 243 phrases), reprise depuis un checkpoint d'époque 10.
2. Version 2 (`afroluo_data_v2.zip`, 11 273 phrases) : les 4 directions ci-dessus.
3. Version 3 : jeux **sans fuite** + glossaire validé. Évaluation stricte sur 715 exemples jamais vus : **BLEU 19,7 — chrF++ 42,1**.

⚠️ Les scores de la v2 (27,4) et de la v3 (19,7) **ne se comparent pas** : ils ne sont pas calculés sur le même test, et le test de la v3 est plus strict. C'est le chiffre de la v3 qu'il faut citer comme performance réelle.

Défauts observés : fuites d'anglais (le modèle de départ est fr/en), répétitions, mots français laissés tels quels. Ces sorties sont filtrées, et le modèle sert de **référence** pour la comparaison avec NLLB.

In [ ]:
import os, json, numpy as np, shutil
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel, MarianTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
    TrainerCallback,
)
import evaluate
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'PAS DE GPU!'}")

DRIVE_FINAL = '/content/drive/MyDrive/AFROLUO_MODELES_FINAUX'

class SaveModelEachEpoch(TrainerCallback):
    """Sauvegarde le vrai modele sur Drive apres chaque epoque."""
    def __init__(self, drive_path, tokenizer):
        self.drive_path = drive_path
        self.tokenizer = tokenizer

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        dest = f"{self.drive_path}_epoch_{epoch}"
        os.makedirs(dest, exist_ok=True)
        model.save_pretrained(dest)
        self.tokenizer.save_pretrained(dest)
        print(f"\nSauvegarde complete epoch {epoch} -> {dest}")

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def load_data(data_dir, direction):
    src, tgt = direction.split('_')
    dir_path = Path(data_dir) / direction
    datasets = {}
    for split in ['train', 'validation', 'test']:
        fp = dir_path / f'{split}.jsonl'
        if not fp.exists(): continue
        raw = load_jsonl(str(fp))
        datasets[split] = Dataset.from_dict({
            'source': [r['translation'][src] for r in raw],
            'target': [r['translation'][tgt] for r in raw],
        })
        print(f"  {split}: {len(datasets[split])}")
    return DatasetDict(datasets)

def preprocess(examples, tokenizer):
    mi = tokenizer(examples['source'], max_length=128, truncation=True, padding=False)
    lb = tokenizer(text_target=examples['target'], max_length=128, truncation=True, padding=False)
    mi['labels'] = lb['input_ids']
    return mi

def compute_metrics(eval_preds, tokenizer):
    bleu = evaluate.load('sacrebleu')
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[[l.strip()] for l in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

def train(direction, data_dir='/content/afroluo_data_v2/'):
    BASE_MODELS = {
        "fr_ew": "Helsinki-NLP/opus-mt-fr-en",
        "ew_fr": "Helsinki-NLP/opus-mt-en-fr",
        "en_ew": "Helsinki-NLP/opus-mt-en-fr",
        "ew_en": "Helsinki-NLP/opus-mt-fr-en",
    }
    drive_path = f"{DRIVE_FINAL}/{direction}"
    output_dir = f"/content/models/{direction}"
    os.makedirs(drive_path, exist_ok=True)

    print(f"\n{'='*50}")
    print(f"ENTRAINEMENT {direction.upper()}")
    print(f"{'='*50}")

    tokenizer = MarianTokenizer.from_pretrained(BASE_MODELS[direction])
    model = MarianMTModel.from_pretrained(BASE_MODELS[direction])

    print("Chargement donnees...")
    dataset = load_data(data_dir, direction)

    tokenized = dataset.map(
        lambda x: preprocess(x, tokenizer),
        batched=True,
        remove_columns=['source', 'target'],
    )

    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=15,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        warmup_steps=200,
        weight_decay=0.01,
        logging_steps=100,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='bleu',
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=128,
        fp16=True,
        learning_rate=5e-5,
        save_total_limit=1,
        report_to='none',
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
        compute_metrics=lambda p: compute_metrics(p, tokenizer),
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            SaveModelEachEpoch(drive_path, tokenizer),
        ],
    )

    trainer.train()

    # Sauvegarde finale garantie
    print("\nSauvegarde finale...")
    final_path = f"{drive_path}/final"
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)

    # Verifie que model.safetensors existe
    if os.path.exists(f"{final_path}/model.safetensors"):
        print(f"model.safetensors present !")
    else:
        model.save_pretrained(final_path)
        tokenizer.save_pretrained(final_path)
        print(f"Sauvegarde forcee !")

    print(f"TERMINE {direction} -> {final_path}")
    return trainer.evaluate(tokenized['test']).get('eval_bleu', 0)

print("Script charge !")

### Lancer l'entraînement Marian
Les données sont dans `afroluo_data_v2.zip` (à téléverser dans Colab puis décompresser dans `/content/`).

In [ ]:
from google.colab import files
uploaded = files.upload()   # choisir afroluo_data_v2.zip
!unzip -q afroluo_data_v2.zip -d /content/

In [ ]:
bleu = train('fr_ew'); print('BLEU final FR->EW :', bleu)

In [ ]:
bleu = train('ew_fr'); print('BLEU final EW->FR :', bleu)

### Essai rapide du modèle Marian (exemples réels de la première version)
| Français | Ewondo proposé |
|---|---|
| Je veux apprendre l'ewondo | Ma vogëlë ewondo |
| Bonne nuit | Mbəmbə alú |
| Je suis heureux | Mënë mvɔé |
| je vais à l'école | makë a ekol |

Certaines sorties sont fausses (« ta photo est jolie » → mot français laissé tel quel), ce qui illustre pourquoi une validation par une locutrice native est indispensable.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
MODEL_PATH = '/content/drive/MyDrive/AFROLUO_MODELES_FINAUX/fr_ew/final'
tokenizer = MarianTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = MarianMTModel.from_pretrained(MODEL_PATH, local_files_only=True).eval()
def traduire(texte):
    inputs = tokenizer(texte, return_tensors="pt", max_length=128, truncation=True)
    out = model.generate(**inputs, num_beams=4, max_length=128)
    return tokenizer.decode(out[0], skip_special_tokens=True)
for f in ["Je veux apprendre l'ewondo", "Bonne nuit", "Je suis heureux"]: print(f, "→", traduire(f))

## 5. Modèle 2 — NLLB-200-distilled-600M affiné
**Idée clé** : NLLB ne connaît pas l'ewondo ; on réutilise le code d'une langue bantoue proche, le lingala (`lin_Latn`), comme étiquette « ewondo ».

**Choix d'entraînement**
- Optimiseur Adafactor, lr 1e-4, warmup 200 pas, label smoothing 0,1, fp16
- batch 8 × accumulation 4 = 32, jusqu'à 4 époques, `max_length` 96
- évaluation et sauvegarde tous les 400 pas ; **meilleur modèle selon chrF++** ; arrêt anticipé (patience 3)
- **reprise après coupure Colab** (`dernier_checkpoint`) : seuls les checkpoints complets sont repris
- collateur personnalisé (`collateur`) qui fournit explicitement `decoder_input_ids` (corrige l'erreur *decoder_input_ids ValueError* rencontrée avec NLLB)

La cellule suivante contient tout le code (fonctions `train`, `compare`, etc.). Attention : elle redéfinit la fonction `train` de Marian ; exécute donc d'abord toute la partie Marian (section 4), puis cette partie.

In [ ]:
# -*- coding: utf-8 -*-
"""AfroLuo - fine-tuning NLLB-200 (600M) francais <-> ewondo, avec le glossaire, SANS fuite, avec reprise apres coupure.
NLLB ne connait pas l'ewondo : on reutilise le code de langue du lingala (lin_Latn, langue bantoue) comme etiquette 'ewondo'.
Cellule 1 : drive.mount ; !pip install -q transformers datasets sacrebleu sentencepiece accelerate
Cellule 2 : exec(open(B + "/train_nllb.py", encoding="utf-8").read()) ; train("fr_ew") ; compare("fr_ew")
Fichiers requis dans afroluo_v3 : data_v4/, glossaire_utilise_1.csv, traduction_francais_ewondo.csv (pour le test des termes)."""
import os, re, csv, json, glob, unicodedata, torch, numpy as np, sacrebleu
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback, MarianMTModel, MarianTokenizer)

B = "/content/drive/MyDrive/afroluo_v3"; DATA = B + "/data_v4"; OUT = "/content/drive/MyDrive/AFROLUO_NLLB"
BASE = "facebook/nllb-200-distilled-600M"; FR, EW = "fra_Latn", "lin_Latn"
MARIAN_V3 = "/content/drive/MyDrive/AFROLUO_MODELES_V3/{d}/final"
nf = lambda s: unicodedata.normalize("NFC", s)
strip = lambda s: "".join(c for c in unicodedata.normalize("NFD", s.lower()) if unicodedata.category(c) != "Mn").replace("ɛ", "e").replace("ɔ", "o").replace("ə", "e")

def glossaire():
    """[(francais, ewondo, page)] depuis glossaire_utilise_1.csv (entrees ambigues ignorees)."""
    out = []
    for r in csv.DictReader(open(B + "/glossaire_utilise_1.csv", encoding="utf-8-sig"), delimiter=";"):
        fr, ew = r["Entrée du lexique"], r["Éwondo"].strip()
        if "pluriel" in fr.lower() or not ew: continue
        fr = re.split(r"\s*[:(/]\s*", fr)[0].strip().lower()
        if fr: out.append((fr, nf(ew), r["Page"]))
    return out

def charger(d, part):
    L = [json.loads(l) for l in open(f"{DATA}/{d}_{part}.jsonl", encoding="utf-8")]
    if part == "train":                                       # le glossaire est ajoute x8 : il pese, sans dominer
        for fr, ew, _ in glossaire():
            for _ in range(8): L.append({"src": fr, "tgt": ew} if d == "fr_ew" else {"src": ew, "tgt": fr})
    return Dataset.from_list(L)

def checkpoint_valide(p):
    return all(os.path.exists(f"{p}/{f}") for f in ("trainer_state.json", "optimizer.pt")) and \
           any(os.path.exists(f"{p}/{w}") for w in ("model.safetensors", "pytorch_model.bin"))
def dernier_checkpoint(dossier):
    for p in sorted(glob.glob(f"{dossier}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True):
        if checkpoint_valide(p): return p
        print("checkpoint incomplet ignore :", p)

def collateur(tok, model):
    """Collateur qui ajoute explicitement decoder_input_ids (labels décalés d'un cran)."""
    base = DataCollatorForSeq2Seq(tok, model=None, label_pad_token_id=-100)
    start = model.config.decoder_start_token_id
    if start is None: start = tok.eos_token_id
    def f(feats):
        b = base(feats)
        lab = b["labels"]
        dec = lab.new_zeros(lab.shape)
        dec[:, 1:] = lab[:, :-1].clone()
        dec[:, 0] = start
        dec[dec == -100] = tok.pad_token_id
        b["decoder_input_ids"] = dec
        return b
    return f

def train(d, epochs=4):
    src, tgt = (FR, EW) if d == "fr_ew" else (EW, FR)
    tok = AutoTokenizer.from_pretrained(BASE, src_lang=src, tgt_lang=tgt)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE); model.generation_config.forced_bos_token_id = tok.convert_tokens_to_ids(tgt)
    enc = lambda b: tok(b["src"], text_target=b["tgt"], max_length=96, truncation=True)
    tr = charger(d, "train").map(enc, batched=True, remove_columns=["src", "tgt"]); va = charger(d, "val").map(enc, batched=True, remove_columns=["src", "tgt"])
    def metrics(p):
        pred, lab = p; pred = pred[0] if isinstance(pred, tuple) else pred
        lab = np.where(lab != -100, lab, tok.pad_token_id); pred = np.where(pred != -100, pred, tok.pad_token_id)
        h = tok.batch_decode(pred, skip_special_tokens=True); r = tok.batch_decode(lab, skip_special_tokens=True)
        return {"bleu": sacrebleu.corpus_bleu(h, [r]).score, "chrf": sacrebleu.corpus_chrf(h, [r], word_order=2).score}
    ck = f"{OUT}/{d}/checkpoints"; os.makedirs(ck, exist_ok=True)
    args = Seq2SeqTrainingArguments(
        output_dir=ck, num_train_epochs=epochs, per_device_train_batch_size=8, gradient_accumulation_steps=4,
        learning_rate=1e-4, optim="adafactor", warmup_steps=200, label_smoothing_factor=0.1, fp16=torch.cuda.is_available(),
        eval_strategy="steps", eval_steps=400, save_strategy="steps", save_steps=400, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
        predict_with_generate=True, generation_max_length=96, generation_num_beams=1, report_to="none", seed=42)
    t = Seq2SeqTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va, processing_class=tok,
                       data_collator=collateur(tok, model), compute_metrics=metrics,
                       callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    rep = dernier_checkpoint(ck); print("REPRISE depuis", rep) if rep else print("Nouveau depart depuis NLLB")
    t.train(resume_from_checkpoint=rep)
    final = f"{OUT}/{d}/final"; os.makedirs(final, exist_ok=True); t.save_model(final); tok.save_pretrained(final)
    assert os.path.exists(final + "/model.safetensors"), "sauvegarde finale incomplete"; print("Modele final :", final)

def trad_nllb(d, textes, bs=16):
    src, tgt = (FR, EW) if d == "fr_ew" else (EW, FR); p = f"{OUT}/{d}/final"
    tok = AutoTokenizer.from_pretrained(p, src_lang=src, tgt_lang=tgt); m = AutoModelForSeq2SeqLM.from_pretrained(p).eval()
    if torch.cuda.is_available(): m = m.half().cuda()
    res = []
    for i in range(0, len(textes), bs):
        b = tok(textes[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=96)
        if torch.cuda.is_available(): b = {k: v.cuda() for k, v in b.items()}
        with torch.no_grad(): g = m.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), num_beams=5, max_length=96, no_repeat_ngram_size=3)
        res += tok.batch_decode(g, skip_special_tokens=True)
    return res

def trad_marian(d, textes, bs=32):
    p = MARIAN_V3.format(d=d); tok = MarianTokenizer.from_pretrained(p); m = MarianMTModel.from_pretrained(p).eval()
    if torch.cuda.is_available(): m = m.cuda()
    res = []
    for i in range(0, len(textes), bs):
        b = tok(textes[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=128)
        if torch.cuda.is_available(): b = {k: v.cuda() for k, v in b.items()}
        with torch.no_grad(): g = m.generate(**b, num_beams=5, max_length=96, no_repeat_ngram_size=3, repetition_penalty=1.3)
        res += tok.batch_decode(g, skip_special_tokens=True)
    return res

def taux_glossaire(sorties_par_phrase, phrases):
    """Pour chaque phrase du CSV contenant un terme du glossaire (comparaison par debut de mot), la sortie contient-elle l'ewondo attendu ?
    Mesure rustique du vocabulaire, PAS de la grammaire ni du sens de la phrase."""
    ok = tot = 0
    for fr, out in zip(phrases, sorties_par_phrase):
        mots = re.findall(r"[a-zà-ÿ]+", strip(fr))
        for g_fr, g_ew, _ in glossaire():
            stem = strip(g_fr)[:max(4, len(strip(g_fr)) - 2)]
            if any(w.startswith(stem) for w in mots):
                tot += 1; ok += strip(g_ew) in strip(out)
    return ok, tot

def compare(d="fr_ew"):
    """Marian v3 (avant) contre NLLB (apres), memes phrases de test jamais vues. Les references viennent des donnees."""
    res = {}
    for nom in (f"{d}_test", f"{d}_test_lexique"):
        ex = [json.loads(l) for l in open(f"{DATA}/{nom}.jsonl", encoding="utf-8")]; src = [x["src"] for x in ex]; ref = [x["tgt"] for x in ex]
        print(f"\n== {nom} ({len(ex)} exemples)")
        for nm, fn in (("Marian v3", trad_marian), ("NLLB", trad_nllb)):
            h = fn(d, src); print(f"  {nm:10s} BLEU {sacrebleu.corpus_bleu(h, [ref]).score:5.1f} | chrF++ {sacrebleu.corpus_chrf(h, [ref], word_order=2).score:5.1f}"); res[(nom, nm)] = h
    fr = [r[0] for r in csv.reader(open(B + "/traduction_francais_ewondo.csv", encoding="utf-8-sig"), delimiter=";") if r and r[0].strip()][1:]
    print("\n== Termes du glossaire retrouves dans les phrases SAFINUM (vocabulaire seulement)")
    for nm, fn in (("Marian v3", trad_marian), ("NLLB", trad_nllb)):
        o, t = taux_glossaire(fn(d, fr), fr); print(f"  {nm:10s} {o}/{t} ({100 * o / max(t, 1):.0f} %)")


### Lancer l'entraînement
Durée : plusieurs dizaines de minutes sur T4. Si la session saute, relancer la même cellule : l'entraînement reprend au dernier checkpoint valide.

In [ ]:
train("fr_ew")

### Si le dossier `final` manque
Il arrive que l'entraînement soit interrompu avant la sauvegarde finale. On peut alors repartir du meilleur checkpoint :

In [ ]:
import glob, shutil
ck = f"{OUT}/fr_ew/checkpoints"
print(sorted(glob.glob(ck + "/checkpoint-*")))
# Exemple : copier le checkpoint voulu en "final"
# best = dernier_checkpoint(ck); shutil.copytree(best, f"{OUT}/fr_ew/final", dirs_exist_ok=True)
# tok = AutoTokenizer.from_pretrained(BASE, src_lang=FR, tgt_lang=EW); tok.save_pretrained(f"{OUT}/fr_ew/final")

## 6. Évaluation : Marian v3 contre NLLB
Mêmes phrases de test jamais vues. Métriques : **BLEU** et **chrF++** (plus fiable pour une langue agglutinante/tonale), plus le taux de termes du glossaire retrouvés (vocabulaire seulement, pas la grammaire).

In [ ]:
compare("fr_ew")

**Résultats retenus** : NLLB (chrF++ 45,1) > Marian v3 (chrF++ 42,1). Les deux restent insuffisants pour traduire seuls une interface financière → d'où l'approche *modèle propose, locutrice valide, dictionnaire applique*.

In [ ]:
# Essai rapide
phrases = ["Envoyer de l'argent", "Montant du transfert", "Frais de transaction", "Confirmer le paiement"]
for f, e in zip(phrases, trad_nllb("fr_ew", phrases)): print(f, "→", e)

## 7. Compression (CTranslate2, int8)
Réduit la taille du modèle pour un usage local (pas dans le navigateur). **Cette étape n'a pas encore été exécutée.**

In [ ]:
!pip install -q ctranslate2

In [ ]:
import subprocess
src_dir = f"{OUT}/fr_ew/final"; out_dir = f"{OUT}/fr_ew/ct2_int8"
subprocess.run(["ct2-transformers-converter","--model",src_dir,"--output_dir",out_dir,"--quantization","int8","--force"], check=True)
print(subprocess.run(["du","-sh",src_dir,out_dir],capture_output=True,text=True).stdout)

In [ ]:
import ctranslate2
from transformers import AutoTokenizer
tr = ctranslate2.Translator(out_dir, device="cpu")
tk = AutoTokenizer.from_pretrained(src_dir, src_lang=FR)
def traduire_ct2(textes):
    toks = [tk.convert_ids_to_tokens(tk.encode(t)) for t in textes]
    res = tr.translate_batch(toks, target_prefix=[[EW]]*len(toks), beam_size=4, no_repeat_ngram_size=3)
    return [tk.decode(tk.convert_tokens_to_ids(r.hypotheses[0][1:])) for r in res]
print(traduire_ct2(["Envoyer de l'argent"]))

## 8. Du modèle à l'extension (méthode de validation)
1. Le modèle traduit les textes de l'interface (`traduire_page.py`) : les chiffres sont masqués puis vérifiés, un texte dont un chiffre change est rejeté, les mots français restants sont remplacés via le glossaire.
2. La locutrice native **Philomène** relit les propositions (fichiers Excel « à valider »).
3. Seules les entrées validées entrent dans `dictionaries/ewondo.json` (`status: valide`, `critique` pour montants/frais/bénéficiaires).
4. L'extension (`content.js`) remplace les textes par règles/dictionnaire, hors ligne, sans modifier le code du site ni les montants, frais ou bénéficiaires.

## 9. Limites et perspectives
- Quelques textes de la page restent en français tant qu'ils ne sont pas validés.
- Orthographe à harmoniser avec la locutrice.
- Perspectives : plus de données, enregistrements audio, modèle compressé embarqué, autres langues avec le même pipeline.